# 33 (PW) — Clean & Transform

**Production workflow, step 2.** Turn raw rows into a queryable Silver table: data-quality fixes, date parts, window features. Grounded in `docs/known_differences.md` and `docs/architecture_guide.md`.

In [ ]:
import os
from dotenv import load_dotenv
from irispark import IrisParkSession

load_dotenv()

# Connection via environment variables (matches examples/basic_usage.py).
# Set IRIS_HOST / IRIS_PORT / IRIS_NAMESPACE / IRIS_USERNAME / IRIS_PASSWORD.
try:
    session = IrisParkSession.builder() \
        .host(os.environ.get("IRIS_HOST", "localhost")) \
        .port(int(os.environ.get("IRIS_PORT", 1972))) \
        .namespace(os.environ.get("IRIS_NAMESPACE", "USER")) \
        .username(os.environ.get("IRIS_USERNAME", "_SYSTEM")) \
        .password(os.environ.get("IRIS_PASSWORD", "SYS")) \
        .getOrCreate()
    print("Connected to IRIS:", session)
except Exception as e:
    print("SKIP: IRIS not reachable -", e)
    session = None

In [ ]:
if session is None:
    raise SystemExit("IRIS not reachable; skipping this notebook.")

## 1. Deliberately dirty source

In [ ]:
import pandas as pd

raw = session.createDataFrame(pd.DataFrame({
    "pedido_id": [1, 2, 2, 3, 4, 5],
    "data": ["2025-01-15", "2025-01-20", "2025-01-20", None, "2025-02-01", "2025-02-10"],
    "estado": ["SP", "RJ", "RJ", "SP", "XX", "MG"],
    "valor": [100.0, None, None, 250.0, 300.0, 90.0],
}))
raw.show()

## 2. Completeness

NULL counts per critical column.

In [ ]:
from irispark.functions import col, count, when

raw.agg(
    count(when(col("data").isNull(), 1)).alias("data_nulls"),
    count(when(col("valor").isNull(), 1)).alias("valor_nulls"),
).show()

## 3. Uniqueness & dedup

Keep the first occurrence of a business key.

In [ ]:
dedup = raw.dropDuplicates(["pedido_id"])
print("raw:", raw.count(), "| dedup:", dedup.count())

## 4. Domain validity

Whitelist states; mark invalid rows instead of failing the whole flow.

In [ ]:
VALIDOS = ["SP", "RJ", "MG", "RS"]
from irispark.functions import when as w, lit

flagged = dedup.withColumn("estado_ok", w(col("estado").isin(VALIDOS), lit("ok")).otherwise(lit("invalid")))
flagged.groupBy("estado_ok").count().show()

## 5. Fix NULLs and invalid values

`fillna` for missing amounts; `when` to repair the invalid state.

In [ ]:
from irispark.functions import when as w, lit

cleaned = flagged \
    .na.fill({"valor": 0.0}) \
    .na.fill({"data": "2025-01-01"}) \
    .withColumn("estado", w(col("estado_ok") == "invalid", lit("DESCONHECIDO")).otherwise(col("estado"))) \
    .drop("estado_ok")
cleaned.show()

## 6. Date parts & derived features

Split the date into year/month/day for easier downstream grouping.

In [ ]:
from irispark.functions import year, month, dayofmonth

with_dates = cleaned.withColumn("ano", year(col("data"))) \
                     .withColumn("mes", month(col("data"))) \
                     .withColumn("dia", dayofmonth(col("data")))
with_dates.show()

## 7. Window feature — running total per state

Cumulative sum over the order history per state.

In [ ]:
from irispark import Window
from irispark.functions import sum as s

w = Window.partitionBy("estado").orderBy("data").rowsBetween(-1, 0)
feat = with_dates.withColumn("running_total", s(col("valor")).over(w))
feat.orderBy("estado", "data").show()

## 8. Materialize the Silver table

In [ ]:
silver = feat.select("pedido_id", "data", "estado", "valor", "ano", "mes", "running_total")
silver.write.mode("overwrite").saveAsTable("pw_silver")
print("silver rows:", session.table("pw_silver").count())

In [ ]:
session.sql("DROP TABLE IF EXISTS pw_silver")
print("dropped pw_silver")

In [ ]:
if session is not None:
    session.close()
    print("Session closed.")